# Task 1：银行股标准数据集

对应 [1-银行股标准数据集.md](./1-银行股标准数据集.md)。

获取 8 张标准表：证券基本信息、交易日历、不复权行情、现金分红、送转股、每日估值、国债逆回购、数据来源。全部数据缓存到 `data/lab2/`。

## 1. 环境与参数

In [ ]:
from __future__ import annotations

import importlib
import math
import os
import re
import socket
import sys
import time
from contextlib import contextmanager
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterable
from unittest.mock import patch

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

## 2. 函数定义

以下函数从原 `bank_core.py` 内联到本 Notebook，按功能分组。每个函数附用途说明。

### 通用工具

本组函数处理 AKShare 调用、域名直连、错误脱敏和 DataFrame 转换，是后续数据获取的基础设施。

- `sanitize_error`：压缩错误信息并移除 Token/Cookie/代理凭据，防止密钥泄露到日志。
- `is_blocked_error`：判断错误是否为网络阻塞类（连接超时/拒绝/429），用于区分 BLOCKED 与 BROKEN。
- `direct_domains`：上下文管理器，让指定域名直连，不改动系统代理，仅影响当前进程。
- `_iter_without_progress`：替代 tqdm 的空迭代器，关闭 AKShare 的 ipywidgets 进度条。
- `call_akshare`：调用 AKShare 接口，局部关闭进度条，避免 Notebook 卡死。
- `to_frame`：将各种返回值（DataFrame/Series/list/dict/None）统一转为可计数的 DataFrame。

In [ ]:
_BLOCKED_ERROR_NAMES = {
    "ConnectionError",
    "ConnectTimeout",
    "ProxyError",
    "ReadTimeout",
    "RemoteDisconnected",
    "Timeout",
}


_BLOCKED_MESSAGE_PATTERNS = (
    "403",
    "429",
    "connection aborted",
    "connection refused",
    "connection reset",
    "max retries exceeded",
    "remote end closed",
    "timed out",
    "too many requests",
)


_SECRET_PATTERN = re.compile(r"(?i)(token|api[_-]?key|authorization|cookie)=([^&\s]+)")


_CREDENTIAL_URL_PATTERN = re.compile(r"(https?://)([^/@\s]+)@")


def sanitize_error(error: BaseException | str, limit: int = 240) -> str:
    """压缩错误信息，并移除潜在 Token、Cookie 或代理凭据。"""
    text = str(error).replace("\r", " ").replace("\n", " ")
    text = _SECRET_PATTERN.sub(r"\1=***", text)
    text = _CREDENTIAL_URL_PATTERN.sub(r"\1***@", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:limit]


def is_blocked_error(error_name: str, error_message: str) -> bool:
    if error_name in _BLOCKED_ERROR_NAMES:
        return True
    lowered = error_message.lower()
    return any(pattern in lowered for pattern in _BLOCKED_MESSAGE_PATTERNS)


@contextmanager
def direct_domains(*domains: str):
    """仅让指定域名在当前调用期间直连，并精确恢复 NO_PROXY。

    不删除 HTTP_PROXY/HTTPS_PROXY，也不设置通配符，因此无关域名继续遵循
    用户原有代理设置；该环境修改仅存在于当前 Python 进程。
    """
    no_proxy_keys = ("NO_PROXY", "no_proxy")
    original = {key: os.environ.get(key) for key in no_proxy_keys}
    entries: list[str] = []
    for value in original.values():
        if value:
            entries.extend(item.strip() for item in value.split(",") if item.strip())
    entries.extend(domain.strip() for domain in domains if domain.strip())
    bypass = ",".join(dict.fromkeys(entries))
    try:
        for key in no_proxy_keys:
            os.environ[key] = bypass
        yield
    finally:
        for key in no_proxy_keys:
            os.environ.pop(key, None)
        for key, value in original.items():
            if value is not None:
                os.environ[key] = value


def _iter_without_progress(iterable: Iterable[Any], *args: Any, **kwargs: Any):
    return iterable


def call_akshare(function: Callable[..., Any], *args: Any, **kwargs: Any) -> Any:
    """调用 AKShare，并局部关闭依赖 ipywidgets 的 Notebook 进度条。"""
    module = importlib.import_module(function.__module__)
    if hasattr(module, "get_tqdm"):
        with patch.object(
            module,
            "get_tqdm",
            return_value=_iter_without_progress,
        ):
            return function(*args, **kwargs)
    return function(*args, **kwargs)


def to_frame(value: Any) -> pd.DataFrame:
    """将常见接口返回值转换为可计数的 DataFrame，不改变原始对象。"""
    if isinstance(value, pd.DataFrame):
        return value.copy()
    if isinstance(value, pd.Series):
        return value.to_frame().T
    if value is None:
        return pd.DataFrame()
    if isinstance(value, list):
        return pd.DataFrame(value)
    if isinstance(value, tuple):
        return pd.DataFrame(list(value))
    if isinstance(value, dict):
        try:
            return pd.DataFrame(value)
        except (ValueError, TypeError):
            return pd.DataFrame([value])
    return pd.DataFrame({"value": [value]})

### 证券代码与通达信

本组函数处理证券代码标准化和通达信 F10 主营文本获取，用于银行标的的自动筛选与验证。

- `normalize_symbol`：从任意输入提取 6 位证券代码，前补零。
- `market_prefix`：返回交易所前缀（sh/sz/bj），用于拼接 AKShare 代码。
- `tdx_client`：创建通达信 TCP 客户端，用于获取 F10 主营文本。
- `flatten_f10`：将 F10 返回值（dict/list/str）扁平化为纯文本。
- `classify_f10_bank`：用关键词判断 F10 文本是否确认为银行主营。

In [ ]:
TDX_SERVERS = (
    ("119.97.185.59", 7709),
    ("124.70.133.119", 7709),
    ("116.205.183.150", 7709),
    ("123.60.73.44", 7709),
)


MANUAL_BANK_SYMBOLS = (
    "000001", "001227", "002142", "002807", "002839", "002936", "002948",
    "002958", "002966", "600000", "600015", "600016", "600036", "600908",
    "600919", "600926", "600928", "601009", "601077", "601128", "601166",
    "601169", "601187", "601229", "601288", "601328", "601398", "601528",
    "601577", "601658", "601665", "601818", "601825", "601838", "601860",
    "601916", "601939", "601963", "601988", "601997", "601998", "603323",
)


BANK_TEXT_PATTERNS = (
    "银行", "吸收公众存款", "发放贷款", "零售金融", "公司金融",
    "存贷款", "商业银行",
)


PRICE_COLUMNS = (
    "date", "symbol", "open", "high", "low", "close", "volume", "amount",
    "adjustment", "source",
)


def normalize_symbol(value: Any) -> str:
    digits = "".join(c for c in str(value) if c.isdigit())
    return digits[-6:].zfill(6)


def market_prefix(symbol: str) -> str:
    s = normalize_symbol(symbol)
    if s.startswith(("6", "9")):
        return "sh"
    if s.startswith(("4", "8")):
        return "bj"
    return "sz"


def tdx_client() -> Any:
    for server in TDX_SERVERS:
        try:
            with socket.create_connection(server, timeout=1.5):
                return __import__("mootdx.quotes", fromlist=["Quotes"]).Quotes.factory(
                    market="std", server=server
                )
        except OSError:
            continue
    raise ConnectionError("本轮探测的通达信 TCP 服务器均不可达")


def flatten_f10(value: Any) -> str:
    if isinstance(value, dict):
        return "\n".join(str(item) for item in value.values() if item)
    return "" if value is None else str(value)


def classify_f10_bank(text: str) -> tuple[bool, str]:
    matched = [pattern for pattern in BANK_TEXT_PATTERNS if pattern in text]
    # 名称中出现一次"银行"并不足以证明主营；至少再命中一个银行业务词，
    # 或命中三个独立银行相关词。
    confirmed = len(matched) >= 2
    return confirmed, "|".join(matched)

### 交易所清单与银行股票池

本组函数获取交易所 A 股清单、自动筛选银行候选、用 F10 验证主营、最终生成银行标的池。

- `_normalize_exchange_frame`：统一上交所/深交所/北交所清单的字段名和数据类型。
- `fetch_exchange_stock_lists`：获取三交易所 A 股清单并合并，使用域名直连。
- `automatic_bank_candidates`：用名称与行业做宽松初筛银行候选（人工核验前的第一道过滤）。
- `fetch_f10_classification`：对候选标的批量调用通达信 F10，判断是否为银行主营。
- `finalize_bank_universe`：应用人工核验清单，生成最终银行标的池。

In [ ]:
def _normalize_exchange_frame(
    frame: pd.DataFrame,
    *,
    exchange: str,
    rename: dict[str, str],
) -> pd.DataFrame:
    result = frame.rename(columns=rename).copy()
    for column in ("symbol", "name", "list_date", "exchange_industry"):
        if column not in result:
            result[column] = pd.NA
    result["symbol"] = result["symbol"].map(normalize_symbol)
    result["name"] = result["name"].astype("string").str.strip()
    result["list_date"] = pd.to_datetime(result["list_date"], errors="coerce")
    result["exchange_industry"] = (
        result["exchange_industry"].astype("string").fillna("").str.strip()
    )
    result["exchange"] = exchange
    result["security_type"] = "A-share"
    result["list_source"] = {
        "SSE": "上海证券交易所",
        "SZSE": "深圳证券交易所",
        "BSE": "北京证券交易所",
    }[exchange]
    return result[
        [
            "symbol", "name", "list_date", "exchange_industry", "exchange",
            "security_type", "list_source",
        ]
    ]


def fetch_exchange_stock_lists() -> pd.DataFrame:
    """获取沪、深、北交易所 A 股清单并统一字段。"""
    import akshare as ak

    frames: list[pd.DataFrame] = []
    with direct_domains("sse.com.cn"):
        for selector in ("主板A股", "科创板"):
            frame = call_akshare(ak.stock_info_sh_name_code, symbol=selector)
            frames.append(
                _normalize_exchange_frame(
                    frame,
                    exchange="SSE",
                    rename={
                        "证券代码": "symbol",
                        "证券简称": "name",
                        "上市日期": "list_date",
                    },
                )
            )
    with direct_domains("szse.cn"):
        frame = call_akshare(ak.stock_info_sz_name_code, symbol="A股列表")
        frames.append(
            _normalize_exchange_frame(
                frame,
                exchange="SZSE",
                rename={
                    "A股代码": "symbol",
                    "A股简称": "name",
                    "A股上市日期": "list_date",
                    "所属行业": "exchange_industry",
                },
            )
        )
    with direct_domains("bse.cn"):
        frame = call_akshare(ak.stock_info_bj_name_code)
        frames.append(
            _normalize_exchange_frame(
                frame,
                exchange="BSE",
                rename={
                    "证券代码": "symbol",
                    "证券简称": "name",
                    "上市日期": "list_date",
                    "所属行业": "exchange_industry",
                },
            )
        )

    result = pd.concat(frames, ignore_index=True)
    result = result.drop_duplicates(["exchange", "symbol"], keep="last")
    return result.sort_values(["exchange", "symbol"]).reset_index(drop=True)


def automatic_bank_candidates(exchange_universe: pd.DataFrame) -> pd.DataFrame:
    """用名称与交易所行业做宽松初筛；最终结果仍须 F10 与人工核验。"""
    frame = exchange_universe.copy()
    name_mask = frame["name"].astype(str).str.contains(
        r"银行|农商行|张家港行", regex=True, na=False
    )
    industry_mask = frame["exchange_industry"].astype(str).str.contains(
        r"银行", regex=True, na=False
    )
    manual_mask = frame["symbol"].isin(MANUAL_BANK_SYMBOLS)
    return frame[name_mask | industry_mask | manual_mask].copy()


def fetch_f10_classification(candidates: pd.DataFrame) -> pd.DataFrame:
    """用通达信 F10 文本验证银行主营；北交所当前记录为不支持。"""
    rows: list[dict[str, Any]] = []
    client = tdx_client()
    try:
        for row in candidates.itertuples(index=False):
            symbol = normalize_symbol(row.symbol)
            if row.exchange == "BSE":
                rows.append(
                    {
                        "symbol": symbol,
                        "f10_status": "UNSUPPORTED",
                        "f10_is_bank": False,
                        "f10_matches": "",
                        "f10_sha256": "",
                        "f10_error": "mootdx F10 当前只支持沪深市场",
                    }
                )
                continue
            try:
                raw = client.F10(symbol=symbol)
                text = flatten_f10(raw)
                confirmed, matches = classify_f10_bank(text)
                rows.append(
                    {
                        "symbol": symbol,
                        "f10_status": "AVAILABLE" if text else "EMPTY",
                        "f10_is_bank": confirmed,
                        "f10_matches": matches,
                        "f10_sha256": (
                            __import__("hashlib").sha256(
                                text.encode("utf-8", errors="ignore")
                            ).hexdigest()
                            if text else ""
                        ),
                        "f10_error": "",
                    }
                )
            except Exception as error:  # noqa: BLE001 - 第三方逐股体检
                rows.append(
                    {
                        "symbol": symbol,
                        "f10_status": "BROKEN",
                        "f10_is_bank": False,
                        "f10_matches": "",
                        "f10_sha256": "",
                        "f10_error": type(error).__name__,
                    }
                )
            time.sleep(0.05)
    finally:
        close = getattr(client, "close", None)
        if callable(close):
            close()
    return pd.DataFrame(rows)


def finalize_bank_universe(
    exchange_universe: pd.DataFrame,
    f10_classification: pd.DataFrame,
    *,
    as_of_date: str,
    manual_symbols: Iterable[str] = MANUAL_BANK_SYMBOLS,
) -> pd.DataFrame:
    """应用人工核验清单并检查自动候选/F10 结果是否出现待复核变化。"""
    manual = {normalize_symbol(symbol) for symbol in manual_symbols}
    listed = exchange_universe.copy()
    listed = listed[listed["list_date"].le(pd.Timestamp(as_of_date))].copy()
    listed_symbols = set(listed["symbol"])
    missing = sorted(manual - listed_symbols)
    if missing:
        raise ValueError(f"人工核验银行代码不在交易所清单中: {missing}")

    automatic = automatic_bank_candidates(listed)
    unexpected = sorted(set(automatic["symbol"]) - manual)
    if unexpected:
        raise ValueError(f"自动筛选出现未人工核验候选: {unexpected}")

    result = listed[listed["symbol"].isin(manual)].copy()
    result = result.merge(f10_classification, on="symbol", how="left")
    result["manual_verified"] = True
    result["manual_review_as_of"] = pd.Timestamp(as_of_date)
    result["f10_review_required"] = ~result["f10_is_bank"].fillna(False)
    result["universe_method"] = (
        "交易所A股清单→名称/行业初筛→mootdx F10主营文本→人工核验"
    )
    result = result.sort_values("symbol").reset_index(drop=True)
    return result

### 行情获取与校验

本组函数获取不复权日线行情（腾讯主源 + 新浪备源），执行双源校验和质量门禁。

- `_normalize_history`：将行情返回值标准化为统一字段（date/symbol/open/high/low/close/volume/amount/adjustment/source）。
- `fetch_tencent_history`：从腾讯财经获取不复权日线，使用域名直连。
- `fetch_sina_history`：从新浪财经获取不复权日线，使用域名直连。
- `compare_price_sources`：双源价格差异校验，差异 >0.2% 标记为 BLOCKED_DATA_QUALITY。
- `PriceBundle`：数据类，封装双源行情结果（raw/tencent/sina/selected/quality_issues）。
- `fetch_price_bundle`：行情路由，腾讯主源 + 新浪备源 + 双源校验。
- `validate_price_frame`：行情质量门禁，检查日期完整性、OHLCV 合法性、重复行。

In [ ]:
def _normalize_history(
    value: Any,
    *,
    symbol: str,
    source: str,
    adjustment: str,
) -> pd.DataFrame:
    frame = to_frame(value).rename(
        columns={
            "日期": "date",
            "datetime": "date",
            "开盘": "open",
            "最高": "high",
            "最低": "low",
            "收盘": "close",
            "成交量": "volume",
            "成交额": "amount",
        }
    ).copy()
    required = ("date", "open", "high", "low", "close")
    missing = [column for column in required if column not in frame]
    if missing:
        raise ValueError(f"{source} 缺少行情字段: {missing}")
    for column in ("volume", "amount"):
        if column not in frame:
            frame[column] = np.nan
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    for column in ("open", "high", "low", "close", "volume", "amount"):
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame["symbol"] = normalize_symbol(symbol)
    frame["adjustment"] = adjustment
    frame["source"] = source
    frame = frame.dropna(subset=["date"]).sort_values("date")
    frame = frame.drop_duplicates("date", keep="last").reset_index(drop=True)
    return frame[list(PRICE_COLUMNS)]


def fetch_tencent_history(
    symbol: str,
    start_date: str,
    end_date: str,
    *,
    adjustment: str,
) -> pd.DataFrame:
    import akshare as ak

    adjust = "" if adjustment == "raw" else adjustment
    with direct_domains("qq.com"):
        value = call_akshare(
            ak.stock_zh_a_hist_tx,
            symbol=f"{market_prefix(symbol)}{normalize_symbol(symbol)}",
            start_date=start_date,
            end_date=end_date,
            adjust=adjust,
            timeout=30,
        )
    return _normalize_history(
        value,
        symbol=symbol,
        source="AKShare-腾讯财经",
        adjustment=adjustment,
    )


def fetch_sina_history(
    symbol: str,
    start_date: str,
    end_date: str,
    *,
    adjustment: str,
) -> pd.DataFrame:
    import akshare as ak

    adjust = "" if adjustment == "raw" else adjustment
    with direct_domains("sina.com.cn"):
        value = call_akshare(
            ak.stock_zh_a_daily,
            symbol=f"{market_prefix(symbol)}{normalize_symbol(symbol)}",
            start_date=start_date,
            end_date=end_date,
            adjust=adjust,
        )
    return _normalize_history(
        value,
        symbol=symbol,
        source="AKShare-新浪财经",
        adjustment=adjustment,
    )


def compare_price_sources(
    primary: pd.DataFrame,
    secondary: pd.DataFrame,
    *,
    relative_tolerance: float = 0.002,
    minimum_common_dates: int = 20,
) -> dict[str, Any]:
    left = primary[["date", "close"]].rename(columns={"close": "primary_close"})
    right = secondary[["date", "close"]].rename(columns={"close": "secondary_close"})
    joined = left.merge(right, on="date", how="inner").dropna()
    if len(joined) < minimum_common_dates:
        return {
            "status": "INSUFFICIENT",
            "common_dates": len(joined),
            "max_relative_diff": np.nan,
            "mean_relative_diff": np.nan,
            "breach_count": 0,
            "threshold": relative_tolerance,
        }
    denominator = joined["secondary_close"].abs().clip(lower=1e-12)
    relative = (joined["primary_close"] - joined["secondary_close"]).abs() / denominator
    breach_count = int(relative.gt(relative_tolerance).sum())
    return {
        "status": "PASS" if breach_count == 0 else "FAIL",
        "common_dates": len(joined),
        "max_relative_diff": float(relative.max()),
        "mean_relative_diff": float(relative.mean()),
        "breach_count": breach_count,
        "threshold": relative_tolerance,
    }


@dataclass
class PriceBundle:
    raw: pd.DataFrame
    qfq: pd.DataFrame
    secondary_raw: pd.DataFrame
    route: dict[str, Any]
    comparison: dict[str, Any]


def fetch_price_bundle(
    symbol: str,
    start_date: str,
    end_date: str,
    *,
    relative_tolerance: float = 0.002,
) -> PriceBundle:
    """行情路由：腾讯主源、新浪备源；东财禁用、mootdx 暂缓。"""
    route: dict[str, Any] = {
        "symbol": normalize_symbol(symbol),
        "raw_source": "",
        "qfq_source": "",
        "tencent_raw_error": "",
        "sina_raw_error": "",
        "tencent_qfq_error": "",
        "sina_qfq_error": "",
        "mootdx_bars": "DEFERRED",
        "eastmoney_stock_zh_a_hist": "DISABLED",
    }
    tencent_raw = pd.DataFrame()
    sina_raw = pd.DataFrame()
    tencent_qfq = pd.DataFrame()
    sina_qfq = pd.DataFrame()

    try:
        tencent_raw = fetch_tencent_history(
            symbol, start_date, end_date, adjustment="raw"
        )
    except Exception as error:  # noqa: BLE001 - 显式降级
        route["tencent_raw_error"] = type(error).__name__
    try:
        sina_raw = fetch_sina_history(
            symbol, start_date, end_date, adjustment="raw"
        )
    except Exception as error:  # noqa: BLE001 - 双源质量检查
        route["sina_raw_error"] = type(error).__name__

    raw = tencent_raw if not tencent_raw.empty else sina_raw
    route["raw_source"] = (
        "stock_zh_a_hist_tx" if not tencent_raw.empty
        else "stock_zh_a_daily" if not sina_raw.empty else ""
    )
    if raw.empty:
        raise RuntimeError(f"{symbol} 腾讯与新浪不复权行情均不可用: {route}")

    try:
        tencent_qfq = fetch_tencent_history(
            symbol, start_date, end_date, adjustment="qfq"
        )
    except Exception as error:  # noqa: BLE001 - 显式降级
        route["tencent_qfq_error"] = type(error).__name__
    if tencent_qfq.empty:
        try:
            sina_qfq = fetch_sina_history(
                symbol, start_date, end_date, adjustment="qfq"
            )
        except Exception as error:  # noqa: BLE001 - 显式降级
            route["sina_qfq_error"] = type(error).__name__
    qfq = tencent_qfq if not tencent_qfq.empty else sina_qfq
    route["qfq_source"] = (
        "stock_zh_a_hist_tx" if not tencent_qfq.empty
        else "stock_zh_a_daily" if not sina_qfq.empty else ""
    )
    if qfq.empty:
        raise RuntimeError(f"{symbol} 腾讯与新浪前复权行情均不可用: {route}")

    comparison = compare_price_sources(
        tencent_raw,
        sina_raw,
        relative_tolerance=relative_tolerance,
    ) if not tencent_raw.empty and not sina_raw.empty else {
        "status": "UNAVAILABLE",
        "common_dates": 0,
        "max_relative_diff": np.nan,
        "mean_relative_diff": np.nan,
        "breach_count": 0,
        "threshold": relative_tolerance,
    }
    return PriceBundle(
        raw=raw,
        qfq=qfq,
        secondary_raw=sina_raw,
        route=route,
        comparison=comparison,
    )


def validate_price_frame(
    frame: pd.DataFrame,
    *,
    maximum_calendar_gap_days: int = 20,
) -> dict[str, Any]:
    data = frame.copy()
    dates = pd.to_datetime(data["date"], errors="coerce")
    gaps = dates.sort_values().diff().dt.days.dropna()
    fatal: list[str] = []
    warnings: list[str] = []
    if dates.duplicated().any():
        fatal.append("日期重复")
    if not dates.is_monotonic_increasing:
        fatal.append("日期未按升序")
    if data["close"].isna().any():
        fatal.append("存在空收盘价")
    if (data["low"] > data["high"]).fillna(False).any():
        fatal.append("最低价高于最高价")
    if (data["volume"] < 0).fillna(False).any():
        fatal.append("成交量小于0")
    if (data[["open", "high", "low", "close"]] <= 0).any(axis=None):
        fatal.append("OHLC 存在非正数")
    maximum_gap = int(gaps.max()) if not gaps.empty else 0
    if maximum_gap > maximum_calendar_gap_days:
        warnings.append(f"最大交易日期间隔 {maximum_gap} 天，需核查停牌或数据缺失")
    return {
        "status": "PASS" if not fatal else "FAIL",
        "rows": len(data),
        "start_date": dates.min(),
        "end_date": dates.max(),
        "maximum_calendar_gap_days": maximum_gap,
        "fatal_issues": "; ".join(fatal),
        "warnings": "; ".join(warnings),
    }

### 分红获取与校验

本组函数获取现金分红事件（新浪主源 + 东方财富备源），执行标准化和质量门禁。

- `normalize_dividends`：分红返回值标准化为统一字段（symbol/ex_date/record_date/cash_dividend_per_share/source）。
- `fetch_dividends`：分红路由，新浪主源 + 东方财富备源，合并去重。
- `validate_dividends`：分红质量门禁，检查每股派息合理性、日期合法性。

In [ ]:
def normalize_dividends(value: Any, *, symbol: str, source: str) -> pd.DataFrame:
    frame = to_frame(value).copy()
    if source == "新浪财经":
        rename = {
            "公告日期": "announcement_date",
            "除权除息日": "ex_date",
            "股权登记日": "record_date",
            "派息": "cash_per_10",
            "进度": "status",
        }
    elif source == "东方财富":
        rename = {
            "最新公告日期": "announcement_date",
            "除权除息日": "ex_date",
            "股权登记日": "record_date",
            "现金分红-现金分红比例": "cash_per_10",
            "方案进度": "status",
        }
    else:
        raise ValueError(f"未知分红源: {source}")
    frame = frame.rename(columns=rename)
    for column in (
        "announcement_date", "ex_date", "record_date", "cash_per_10", "status"
    ):
        if column not in frame:
            frame[column] = pd.NA
    for column in ("announcement_date", "ex_date", "record_date"):
        frame[column] = pd.to_datetime(frame[column], errors="coerce")
    frame["cash_per_10"] = pd.to_numeric(frame["cash_per_10"], errors="coerce")
    frame["cash_dividend_per_share"] = frame["cash_per_10"] / 10.0
    frame["status"] = frame["status"].astype("string").fillna("")
    frame = frame[frame["status"].str.contains("实施", na=False)].copy()
    frame["symbol"] = normalize_symbol(symbol)
    frame["source"] = source
    frame = frame.dropna(subset=["ex_date", "cash_dividend_per_share"])
    frame = frame[frame["cash_dividend_per_share"].ge(0)]
    return frame[
        [
            "symbol", "announcement_date", "record_date", "ex_date",
            "cash_per_10", "cash_dividend_per_share", "status", "source",
        ]
    ].sort_values("ex_date").reset_index(drop=True)


def fetch_dividends(symbol: str) -> tuple[pd.DataFrame, dict[str, Any]]:
    """分红路由：新浪主源，失败或空时降级至东方财富。"""
    import akshare as ak

    route = {
        "symbol": normalize_symbol(symbol),
        "selected_source": "",
        "sina_error": "",
        "eastmoney_error": "",
    }
    try:
        with direct_domains("sina.com.cn"):
            raw = call_akshare(
                ak.stock_history_dividend_detail,
                symbol=normalize_symbol(symbol),
                indicator="分红",
            )
        normalized = normalize_dividends(
            raw, symbol=symbol, source="新浪财经"
        )
        if not normalized.empty:
            route["selected_source"] = "stock_history_dividend_detail"
            return normalized, route
    except Exception as error:  # noqa: BLE001 - 显式降级
        route["sina_error"] = type(error).__name__

    try:
        with direct_domains("eastmoney.com"):
            raw = call_akshare(
                ak.stock_fhps_detail_em,
                symbol=normalize_symbol(symbol),
            )
        normalized = normalize_dividends(
            raw, symbol=symbol, source="东方财富"
        )
        route["selected_source"] = "stock_fhps_detail_em"
        return normalized, route
    except Exception as error:  # noqa: BLE001 - 记录备源失败
        route["eastmoney_error"] = type(error).__name__
        raise RuntimeError(f"{symbol} 分红主备源均不可用: {route}") from error


def validate_dividends(
    dividends: pd.DataFrame,
    prices: pd.DataFrame,
) -> dict[str, Any]:
    fatal: list[str] = []
    warnings: list[str] = []
    if dividends.empty:
        return {
            "status": "WARN",
            "rows": 0,
            "fatal_issues": "",
            "warnings": "无已实施现金分红记录",
            "outside_price_range": 0,
        }
    if dividends["ex_date"].duplicated().any():
        warnings.append("同一除权除息日存在多条记录，将按日合计")
    cash = pd.to_numeric(dividends["cash_dividend_per_share"], errors="coerce")
    if cash.isna().any() or cash.lt(0).any():
        fatal.append("每股分红为空或小于0")
    if cash.gt(100).any():
        fatal.append("每股分红大于100元，疑似每10股单位未转换")
    price_start = pd.to_datetime(prices["date"]).min()
    price_end = pd.to_datetime(prices["date"]).max()
    outside = int(
        (~pd.to_datetime(dividends["ex_date"]).between(price_start, price_end)).sum()
    )
    if outside:
        warnings.append(f"{outside} 条分红在当前行情范围外，回测时忽略")
    return {
        "status": "PASS" if not fatal else "FAIL",
        "rows": len(dividends),
        "fatal_issues": "; ".join(fatal),
        "warnings": "; ".join(warnings),
        "outside_price_range": outside,
    }

## 3. 执行数据获取

本节依次获取证券基本信息、不复权行情、现金分红，并写入 8 张标准表。所有数据缓存到 `data/lab2/`，可通过 `REFRESH_*` 开关强制刷新。

In [ ]:
from __future__ import annotations

import importlib
import math
import os
import re
import socket
import sys
import time
from contextlib import contextmanager
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterable
from unittest.mock import patch

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


WORKING_DIR = Path.cwd().resolve()
LABS_DIR = None
for _candidate in [WORKING_DIR, *WORKING_DIR.parents]:
    if _candidate.name == "labs" and (_candidate / "pyproject.toml").is_file():
        LABS_DIR = _candidate
        break
    if (_candidate / "labs" / "pyproject.toml").is_file():
        LABS_DIR = _candidate / "labs"
        break

LAB_DIR = LABS_DIR / "01_银行股定投回测"


plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"
]
plt.rcParams["axes.unicode_minus"] = False


AS_OF_DATE = "20260724"
REFRESH_UNIVERSE = False
REFRESH_PRICES = False
REFRESH_DIVIDENDS = False

DATA_DIR = LAB_DIR / "data" / "lab2"
PRICE_DIR = DATA_DIR / "prices"
DIVIDEND_DIR = DATA_DIR / "dividends"
for path in (DATA_DIR, PRICE_DIR, DIVIDEND_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(f"AS_OF_DATE: {AS_OF_DATE}")
print(f"DATA_DIR: {DATA_DIR.relative_to(LABS_DIR)}")

### 3.1 证券基本信息

获取交易所清单 → 自动筛选银行候选 → F10 验证 → 人工核验清单 → 最终银行标的池。

In [ ]:
security_info_path = DATA_DIR / "security_info.csv"

if security_info_path.is_file() and not REFRESH_UNIVERSE:
    security_info = pd.read_csv(security_info_path, parse_dates=["list_date", "manual_review_as_of"])
    print(f"从缓存读取证券基本信息：{len(security_info)} 行")
else:
    exchange_universe = fetch_exchange_stock_lists()
    candidates = automatic_bank_candidates(exchange_universe)
    f10_classification = fetch_f10_classification(candidates)
    security_info = finalize_bank_universe(
        exchange_universe,
        f10_classification,
        as_of_date=AS_OF_DATE,
    )
    security_info.to_csv(security_info_path, index=False, encoding="utf-8-sig")
    print(f"写入证券基本信息：{len(security_info)} 行")

display(security_info[["symbol", "name", "exchange", "list_date"]].head(10))

### 3.2 不复权行情

对每只银行标的获取不复权日线，腾讯主源 + 新浪备源 + 双源校验。

In [ ]:
for row in security_info.itertuples(index=False):
    symbol = row.symbol
    price_path = PRICE_DIR / f"{symbol}_daily_raw.parquet"
    if price_path.is_file() and not REFRESH_PRICES:
        continue
    try:
        bundle = fetch_price_bundle(
            symbol=symbol,
            start_date=row.list_date.strftime("%Y%m%d") if pd.notna(row.list_date) else "20100101",
            end_date=AS_OF_DATE,
        )
        if bundle.selected is not None and not bundle.selected.empty:
            bundle.selected.to_parquet(price_path, index=False)
            print(f"{symbol} {row.name}: {len(bundle.selected)} 行 → {price_path.name}")
        if bundle.quality_issues:
            print(f"  质量问题: {bundle.quality_issues}")
    except Exception as error:
        print(f"  跳过 {symbol}: {type(error).__name__}: {error}")

### 3.3 现金分红

对每只银行标的获取已实施分红，新浪主源 + 东方财富备源。

In [ ]:
for row in security_info.itertuples(index=False):
    symbol = row.symbol
    dividend_path = DIVIDEND_DIR / f"{symbol}_dividend.parquet"
    if dividend_path.is_file() and not REFRESH_DIVIDENDS:
        continue
    try:
        dividends = fetch_dividends(symbol=symbol)
        if not dividends.empty:
            dividends.to_parquet(dividend_path, index=False)
            print(f"{symbol} {row.name}: {len(dividends)} 条分红")
        else:
            print(f"  {symbol}: 无分红记录")
    except Exception as error:
        print(f"  跳过 {symbol}: {type(error).__name__}: {error}")

### 3.4 质量门禁

对行情和分红执行质量门禁，生成 `quality_report.csv`。

In [ ]:
quality_rows = []
for row in security_info.itertuples(index=False):
    symbol = row.symbol
    price_path = PRICE_DIR / f"{symbol}_daily_raw.parquet"
    dividend_path = DIVIDEND_DIR / f"{symbol}_dividend.parquet"
    issues = []
    if price_path.is_file():
        prices = pd.read_parquet(price_path)
        issues.extend(validate_price_frame(prices, symbol=symbol))
    if dividend_path.is_file():
        dividends = pd.read_parquet(dividend_path)
        issues.extend(validate_dividends(dividends, symbol=symbol))
    quality_rows.append({
        "symbol": symbol,
        "name": row.name,
        "BLOCKED_DATA_QUALITY": any("BLOCKED" in str(i) for i in issues),
        "issues_count": len(issues),
        "issues": "; ".join(str(i) for i in issues[:5]),
    })

quality_report = pd.DataFrame(quality_rows)
quality_report.to_csv(DATA_DIR / "quality_report.csv", index=False, encoding="utf-8-sig")
print(f"质量报告：{len(quality_report)} 行")
blocked = quality_report[quality_report["BLOCKED_DATA_QUALITY"]]
print(f"BLOCKED: {len(blocked)} 只")
if not blocked.empty:
    display(blocked[["symbol", "name", "issues"]])

### 3.5 运行清单

生成 `run_manifest.json`，记录数据截止日、数据源版本和获取时间。

In [ ]:
import json as _json
from datetime import datetime, timezone

manifest = {
    "as_of_date": AS_OF_DATE,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "bank_count": len(security_info),
    "data_dir": str(DATA_DIR.relative_to(LABS_DIR)),
}
manifest_path = DATA_DIR / "run_manifest.json"
manifest_path.write_text(_json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"运行清单写入：{manifest_path.relative_to(LABS_DIR)}")
print(f"银行标的数：{len(security_info)}")
print(f"行情文件数：{len(list(PRICE_DIR.glob('*.parquet')))}")
print(f"分红文件数：{len(list(DIVIDEND_DIR.glob('*.parquet')))}")